# Native JSONL and explicit OpenAI mapping

Store one versioned `Conversation.to_json()` record per line. Loading preserves message order and payloads, including malformed tool-argument text. Corpus iteration, error policy, and external dataset integrations belong to the caller.


In [ ]:
from collections import Counter
from pathlib import Path
from tempfile import TemporaryDirectory

from pydantic import ValidationError

from apertus_common import (
    Conversation,
    Document,
    Host,
    OpenAIMessage,
    Reply,
    SystemPrompt,
    User,
    Wait,
    from_openai,
    to_openai,
)

system = SystemPrompt.build(behavior="Be concise.")
records = [
    Conversation(
        system=system, items=[User(payload="Hi"), Reply(payload="Hello"), Wait()]
    ),
    Conversation(
        system=system, items=[User(payload="First"), Host(payload="Later input")]
    ),
]

with TemporaryDirectory(prefix="apertus-dataset-") as temporary:
    path = Path(temporary) / "conversations.jsonl"
    path.write_text(
        "".join(record.to_json() + "\n" for record in records), encoding="utf-8"
    )
    restored: list[Conversation] = []
    findings: Counter[str] = Counter()
    with path.open(encoding="utf-8") as source:
        for number, line in enumerate(source, start=1):
            try:
                record = Conversation.from_json(line)
            except ValidationError as error:
                print(f"Line {number}: {error}")
                continue
            report = record.check()
            findings.update(finding.rule for finding in report.violations)
            findings.update(
                f"unevaluated:{finding.rule}" for finding in report.unevaluated
            )
            restored.append(record)
    assert restored == records
    assert not findings
    assert [item.type for item in restored[1].items] == ["user", "host"]

## Documents require provenance

The library prepares post-training data. Every document declares its supplying party (`from`) and citation ID (`id`); records without them fail the check.

In [ ]:
missing = Conversation(system=system, items=[Document(payload="Source text")])
assert not missing.check().ok
cited = Conversation(
    system=system,
    items=[Document(payload="Source text", from_="user", id="d1")],
)
report = cited.check()
assert report.ok and report.complete
assert Conversation.from_json(cited.to_json()) == cited

## OpenAI mappings return values, diagnostics, and context

Use `.value` for the mapped data. Preserve `.context` when continuing tool conversations or compacting history. The default mapper rejects unsupported or lossy conversions; explicit policies allow particular documented losses.


In [ ]:
messages: list[OpenAIMessage] = [
    {"role": "system", "content": "Be concise."},
    {"role": "user", "content": "Hello"},
    {"role": "assistant", "content": "Hi"},
]
mapped = from_openai(messages, reasoning_effort="medium")
assert mapped.value.check().ok
assert not mapped.diagnostics
request = to_openai(mapped.value, context=mapped.context)
assert request.value.get("messages") == messages
assert request.value.get("reasoning_effort") == "medium"
assert Conversation.from_json(mapped.value.to_json()) == mapped.value
print(request.value)

Keeping native conversation JSON in a text column avoids imposing an external table schema on the message union. OpenAI conversion is a supported subset, not a replacement for native storage. Tokenizer artifacts are acquired separately and are unnecessary for JSONL processing.
